# 00 - Profiling & Setup

**Lädt den Datensatz, repliziert die ZEW-Eckwerte und prüft die Konsistenz der Lade-Pipeline, der Smoke-Test, auf dem alle folgenden Notebooks aufbauen.**

Essay-Bezug: Executive Summary, Abschnitt 1 (Leitfrage und Vorgehen), Abschnitt 2 (Methodik im Überblick)

## Leitfrage

Im Mai 2025 hat die Bundesregierung das Bundesministerium für Digitales und Staatsmodernisierung (BMDS) eingerichtet, eine institutionelle Antwort auf die seit Jahren wiederholte Forderung nach zentraler Steuerung der digitalen Aktivitäten des Bundes. Digitalverbände, Bundesrechnungshof und zuletzt auch das Zentrum für Europäische Wirtschaftsforschung (ZEW) [TODO: konkretes Zitat aus ZEW-Studie nachtragen] fordern zudem ein „Digitalbudget", das den Digitalhaushalt erstmals einheitlich darstellt und steuerbar macht.

**Leitfrage.** Was kann das BMDS tatsächlich steuern, was nicht? Welche strukturellen Grenzen ergeben sich aus der heutigen Verteilung und Logik der digitalen Mittel? Beide Vorhaben, BMDS-Steuerung und Digitalbudget, beruhen auf der bislang kaum geprüften Annahme, dass der Digitalhaushalt überhaupt eine handhabbare, zentral fassbare Größe ist. Diese Annahme prüfen wir in den folgenden Notebooks empirisch.

[TODO Schreib-Person: Anschluss an die laufende politische Debatte aktualisieren, sobald die ZEW-Studie und die zwei Agora-Policy-Papers durchgelesen sind. Lese-Tabelle in `docs/bereits_gesagt_vs_luecke.md` fortlaufend pflegen.]

## Story-Arc: drei Akte

1. **Wo liegt das Geld?** Notebook 01 (H1): Konzentration über Ressorts, Aufgaben-Profile, BMDS-Bündelungs-Szenarien
2. **Was passiert damit?** Notebook 02 (H2): eigene Investition vs. Investitions-Transfer, Treiber, EP-60-Spin-off, Verpflichtungsermächtigungen
3. **Wie wird darüber geredet?** Notebook 03 (H4): politisches Rebranding, Buzzword-Check

Notebook 99 führt die drei Akte zur politischen Implikation zusammen: Klassifikations- und Transparenzhoheit vs. operative Steuerungshoheit über die Mittel.

**Vorgehen.** Wir nutzen den erstmals öffentlich verfügbaren Datensatz von ZEW Mannheim und Agora Digitale Transformation, 21.358 Haushaltstitel über die Jahre 2019, 2021, 2023 und 2024, jeweils mit ausgewiesenem digitalen Volumen. Dieses Notebook ist der Einstieg: Es lädt den Datensatz, repliziert die publizierten Eckwerte und prüft die Konsistenz der Pipeline.

[TODO Schreib-Person: einmal komplett gegenlesen. Stand: Phase 2 abgeschlossen, externe Validierung der drei großen Transfer-Posten steht noch aus.]

## Methodik im Überblick

**Datenbasis.** Digitalhaushalt Open Data (ZEW/Agora, Stand 2025): 21.358 Beobachtungen über 6.240 unique Titel-IDs, Granularität Haushaltstitel × Jahr. Drei Geldspalten in Tausend Euro: `soll` (Gesamt-Sollansatz), `digi_soll_eng` (eng abgegrenzter Digital-Anteil), `digi_soll_weit` (weit abgegrenzter Anteil inklusive Mischausgaben).

**Primäre Abgrenzung.** Alle Hauptbefunde dieser Studie beruhen auf der engen Abgrenzung `digi_soll_eng`. Die weite Abgrenzung (`digi_soll_weit`) dient ausschließlich als Robustheitscheck.

**Was die folgenden Zellen prüfen:** Lädt `load()` den Datensatz korrekt (Zeilen, Jahre, IDs)? Stimmen unsere Aggregate mit den publizierten ZEW-Eckwerten überein? Sind die Geld-Spalten in sich konsistent? Wie stabil ist das Titel-Panel über die Jahre? Wie groß ist die Spreizung zwischen enger und weiter Abgrenzung?

**Pre-Mortem.** Was würde diese Grundlage töten? Eine Replikations-Abweichung über 0,3 Prozent, Verletzungen der Geld-Spalten-Logik (`eng > weit` oder `weit > soll`) oder doppelte `(id, jahr)`-Kombinationen. Tritt eines davon auf, ist die Lade-Pipeline gebrochen, Stop und suchen, bevor irgendeine inhaltliche Aussage getroffen wird.

In [1]:
"""00 - Profiling: Replikation der ZEW-Eckwerte und Konsistenzpruefung.

Phase-0-Output. Laeuft in unter 10 Sekunden, dient als Smoke-Test der
gesamten Pipeline. Fuer Jupyter Lab: als .py mit jupytext oeffnen oder
direkt python ausfuehren.

    python notebooks/00_profiling.py
"""

'00 - Profiling: Replikation der ZEW-Eckwerte und Konsistenzpruefung.\n\nPhase-0-Output. Laeuft in unter 10 Sekunden, dient als Smoke-Test der\ngesamten Pipeline. Fuer Jupyter Lab: als .py mit jupytext oeffnen oder\ndirekt python ausfuehren.\n\n    python notebooks/00_profiling.py\n'

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, '..')  # repo root, relativ zum notebooks/-Verzeichnis
from src.load import load

import pandas as pd

In [3]:
df = load()  # nutzt Repo-Root-Default aus src.load
print(f"Zeilen: {len(df):,}")
print(f"Jahre: {sorted(df['jahr'].unique())}")
print(f"Unique Titel-IDs: {df['id'].nunique():,}\n")

Zeilen: 21,358
Jahre: [np.int64(2019), np.int64(2021), np.int64(2023), np.int64(2024)]
Unique Titel-IDs: 6,240



**Befund.** Die Pipeline lädt den vollständigen Datensatz über die vier verfügbaren Jahre (2020 und 2022 fehlen). Zeilenzahl, Jahresliste und Zahl eindeutiger Titel-IDs müssen mit `DOC.md` Abschnitt 2.1 übereinstimmen, weicht hier etwas ab, ist bereits der Datensatz-Stand falsch, nicht erst die Aggregation.

Nächster Schritt: Stimmen die aggregierten Digital-Volumina mit den publizierten ZEW-Eckwerten überein?

In [4]:
#| label: tbl-replikation
print("=== Replikation ZEW-Eckwerte (digi_soll_eng in Mrd. EUR) ===")
agg = df.groupby('jahr').agg(
    soll_mrd=('soll', lambda x: x.sum() / 1e6),
    eng_mrd=('digi_soll_eng', lambda x: x.sum() / 1e6),
    weit_mrd=('digi_soll_weit', lambda x: x.sum() / 1e6),
    n_titel=('id', 'count'),
).round(2)
print(agg)

erwartet = {2019: 8.5, 2021: 16.6, 2023: 19.2, 2024: 17.9}
print("\nAbweichung digi_soll_eng (Replikation - erwartet):")
for j, e in erwartet.items():
    actual = agg.loc[j, 'eng_mrd']
    delta = actual - e
    print(f"  {j}: {actual:.2f} vs. {e}  Delta = {delta:+.2f} Mrd ({delta/e*100:+.1f}%)")

=== Replikation ZEW-Eckwerte (digi_soll_eng in Mrd. EUR) ===
      soll_mrd  eng_mrd  weit_mrd  n_titel
jahr                                      
2019    356.40     8.51      9.64     5015
2021    572.73    16.55     17.79     5299
2023    461.21    19.19     20.54     5486
2024    476.81    17.94     19.09     5558

Abweichung digi_soll_eng (Replikation - erwartet):
  2019: 8.51 vs. 8.5  Delta = +0.01 Mrd (+0.1%)
  2021: 16.55 vs. 16.6  Delta = -0.05 Mrd (-0.3%)
  2023: 19.19 vs. 19.2  Delta = -0.01 Mrd (-0.1%)
  2024: 17.94 vs. 17.9  Delta = +0.04 Mrd (+0.2%)


**Befund.** Die Replikation der `digi_soll_eng`-Aggregate weicht für alle vier Jahre um weniger als 0,3 Prozent von den publizierten ZEW-Eckwerten ab. Damit ist die Lade- und Aggregationslogik (`load()`, NA-Behandlung als 0, enge Abgrenzung) als Grundlage aller folgenden Hypothesen-Notebooks validiert, und der Maßstab gesetzt: Weicht eine spätere Replikation plötzlich stärker ab, ist die Pipeline gebrochen.

Nächster Schritt: Sind die Geld-Spalten auch intern konsistent, keine Logik-Verletzungen, keine Duplikate?

In [5]:
print("\n=== Konsistenzpruefungen ===")
viol_ew = (df['digi_soll_eng'] > df['digi_soll_weit']).sum()
# weit > soll ist trivial verletzt, wenn soll < 0 (Globale Minderausgaben).
# Pruefen nur fuer nicht-negative soll-Werte:
viol_ws = ((df['digi_soll_weit'] > df['soll']) & (df['soll'] >= 0)).sum()
dup = df.duplicated(subset=['id', 'jahr']).sum()
neg_soll = (df['soll'] < 0).sum()
print(f"  Verletzungen eng > weit: {viol_ew}")
print(f"  Verletzungen weit > soll (ohne Globale Minderausgaben): {viol_ws}")
print(f"  Doppelte (id, jahr): {dup}")
print(f"  Negative soll (Globale Minderausgaben): {neg_soll}")


=== Konsistenzpruefungen ===
  Verletzungen eng > weit: 0
  Verletzungen weit > soll (ohne Globale Minderausgaben): 0
  Doppelte (id, jahr): 0
  Negative soll (Globale Minderausgaben): 121


**Befund.** Keine Verletzung der Geld-Spalten-Logik (`digi_soll_eng <= digi_soll_weit <= soll`) und keine doppelten `(id, jahr)`-Kombinationen. Die negativen `soll`-Werte sind „Globale Minderausgaben"  Haushaltstechnik, keine Anomalie, und durchweg nicht-digital klassifiziert.

Nächster Schritt: Wie stabil ist das Titel-Panel über die vier Jahre, eine Voraussetzung für „stabiler Kern"-Aussagen in späteren Notebooks?

In [6]:
print("\n=== ID-Stabilitaet ueber Jahre ===")
id_jahr = df.groupby('id')['jahr'].nunique()
print(f"  Titel-IDs gesamt: {df['id'].nunique():,}")
for k in [4, 3, 2, 1]:
    n = (id_jahr == k).sum()
    pct = n / df['id'].nunique() * 100
    print(f"  in {k} Jahr(en): {n:,} ({pct:.1f}%)")


=== ID-Stabilitaet ueber Jahre ===
  Titel-IDs gesamt: 6,240
  in 4 Jahr(en): 4,449 (71.3%)
  in 3 Jahr(en): 511 (8.2%)
  in 2 Jahr(en): 749 (12.0%)
  in 1 Jahr(en): 531 (8.5%)


**Befund.** Gut zwei Drittel aller Titel-IDs tauchen in allen vier Jahren auf, ein stabiler Panel-Kern, der Trendvergleiche auf Titel-Ebene erlaubt. Der Rest sind ID-Wechsel, die innerhalb der üblichen Aggregationseinheiten (Einzelplan, Kapitel, Funktion) absorbiert werden. Deshalb ist der Aggregations-Filter der Default für Trendvergleiche in H1 und H2; der Panel-Filter (`panel_titel()`) bleibt die Ausnahme für gezielte „stabiler Kern"-Aussagen.

Nächster Schritt: Wie groß ist die Differenz zwischen enger und weiter Digital-Abgrenzung, und verändert sie sich über die Zeit?

In [7]:
print("\n=== Eng-Weit-Spreizung des Gesamthaushalts ===")
sp = df.groupby('jahr').agg(
    eng=('digi_soll_eng', lambda x: x.sum() / 1e6),
    weit=('digi_soll_weit', lambda x: x.sum() / 1e6),
)
sp['delta'] = sp['weit'] - sp['eng']
sp['delta_pct'] = (sp['delta'] / sp['weit'] * 100).round(1)
print(sp.round(2))


=== Eng-Weit-Spreizung des Gesamthaushalts ===
        eng   weit  delta  delta_pct
jahr                                
2019   8.51   9.64   1.13       11.7
2021  16.55  17.79   1.25        7.0
2023  19.19  20.54   1.35        6.6
2024  17.94  19.09   1.15        6.0


**Befund.** Die Spreizung zwischen enger und weiter Abgrenzung sinkt von 11,7 Prozent (2019) auf 6,0 Prozent (2024), die Haushaltssprache wird über die Jahre eindeutiger digital, Mischausgaben verlieren an Gewicht. Ein kleiner, aber konsistenter Befund am Rande, der in Notebook 03 als Spin-off wieder auftaucht.

Damit ist die Pipeline geprüft. Die folgenden Notebooks bauen direkt auf `load()` auf.

## Kernbefund und weiter

**Die Lade-Pipeline repliziert die ZEW-Eckwerte auf unter 0,3 Prozent Abweichung, ist intern konsistent und bietet mit einem über 70 Prozent Panel-stabilen Titel-Bestand eine tragfähige Grundlage für Trendvergleiche.** Damit ist sichergestellt, dass alle folgenden inhaltlichen Aussagen auf einer geprüften Datenbasis stehen.

**Weiter zu Notebook 01** Akt 1: Wo liegt das Geld? Konzentration, Ressort-Profile und BMDS-Bündelungs-Szenarien.

**Limitation dieser Analyse.** Geprüft wird die technische Konsistenz der Pipeline, nicht die inhaltliche Validität der ZEW-Klassifikation selbst, diese bleibt eine Setzung, die in Notebook 99 unter Limitationen eingeordnet wird.